# Chapitre 7 — Comprendre le calcul d'un réseau de neurones · Corrigé commenté

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch07_corrige.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Chaque réponse commence par le raisonnement sur papier, puis montre son équivalent en Python.

**Sommaire.** Données · Les calculs des leçons 7.1 à 7.3 · Atelier facultatif (douze messages) · Exercices 7.1 à 7.6

**Avant d'ouvrir Jupyter :** faites d'abord à la main le trajet aller de M7 (leçon 7.1) et la dérivée par rapport à $w_{11}$ (leçon 7.2). Les cellules ci-dessous servent à **vérifier** ces calculs.

**Repères :** indigo = calcul vers l'avant ; magenta en tirets = trajet du gradient ; trait épais ambre = poids étudié $w_{11}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repères du livre : calcul vers l'avant en indigo, gradient en magenta (tirets), poids étudié en ambre épais
AVANT, RETOUR, POIDS_ETUDIE = '#3949ab', '#c2185b', '#f9a825'


def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def perte(y, p):
    # perte logarithmique d'un message (équation 5.6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)))


def aller(P, x):
    # Trajet aller (équations 7.1 et 7.2) : renvoie un dictionnaire z1, a1, z2, a2, z, p
    z1 = P['w11'] * x[0] + P['w12'] * x[1] + P['b1']
    z2 = P['w21'] * x[0] + P['w22'] * x[1] + P['b2']
    a1, a2 = sigmoide(z1), sigmoide(z2)
    z = P['v1'] * a1 + P['v2'] * a2 + P['c']
    return {'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2, 'z': z, 'p': sigmoide(z)}


def dessiner(P, x, retour=False):
    # Le schéma du livre, en plus simple : indigo pour l'aller, magenta pour le retour, w11 en ambre épais
    v = aller(P, x)
    pos = {'x1': (0, 3), 'x2': (0, 0), 'n1': (3, 3), 'n2': (3, 0), 's': (6, 1.5)}
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for a, b, nom in (('x1', 'n1', 'w11'), ('x2', 'n1', 'w12'), ('x1', 'n2', 'w21'), ('x2', 'n2', 'w22'),
                      ('n1', 's', 'v1'), ('n2', 's', 'v2')):
        (xa, ya), (xb, yb) = pos[a], pos[b]
        if nom == 'w11':
            ax.plot([xa, xb], [ya, yb], color=POIDS_ETUDIE, linewidth=7, zorder=1)
        ax.annotate('', xy=(xb - 0.35, yb + (ya - yb) * 0.12), xytext=(xa + 0.3, ya),
                    arrowprops=dict(arrowstyle='-|>', color=AVANT), zorder=2)
        t = 0.3 if a[0] == 'x' and b != ('n1' if a == 'x1' else 'n2') else 0.5
        ax.text(xa + t * (xb - xa), ya + t * (yb - ya) + 0.15, f"{nom} = {P[nom]:g}" + (' (poids étudié)' if nom == 'w11' else ''),
                color=AVANT, fontsize=8, ha='center', bbox=dict(facecolor='white', edgecolor='none', alpha=0.85))
    textes = {'x1': f"x1 = {x[0]}", 'x2': f"x2 = {x[1]}",
              'n1': f"N1\nz1 = {v['z1']:.3f}\na1 = {v['a1']:.4f}", 'n2': f"N2\nz2 = {v['z2']:.3f}\na2 = {v['a2']:.4f}",
              's': f"sortie\nz = {v['z']:.3f}\np̂ = {v['p']:.3f}"}
    for k, (cx, cy) in pos.items():
        ax.text(cx, cy, textes[k], ha='center', va='center', fontsize=8,
                bbox=dict(boxstyle='circle', facecolor='white', edgecolor=AVANT if k[0] != 'x' else 'grey'))
    if retour:
        for (x0, y0), (x1, y1), lab in (((6.6, 2.4), (3.5, 3.6), '② v1'), ((3.0, 3.9), (1.4, 3.4), '③ a1(1 − a1), ④ x1')):
            ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle='-|>', color=RETOUR, linestyle='--', connectionstyle='arc3,rad=0.3'))
            ax.text((x0 + x1) / 2, max(y0, y1) + 0.35, lab, color=RETOUR, fontsize=8, ha='center')
        ax.text(6.6, 2.7, '① p̂ − y', color=RETOUR, fontsize=8)
    ax.set_xlim(-0.8, 7.4); ax.set_ylim(-0.8, 4.4); ax.axis('off'); plt.show()

## Données

Les identifiants D7.1, D7.E1 et D5.1 sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 7, recopiées des fichiers CSV du livre (ne pas modifier)

# D7.1 — les neuf paramètres du réseau des leçons 7.1 à 7.3
reseau = {'w11': 0.5, 'w12': 0.5, 'b1': -2, 'w21': 1, 'w22': -1, 'b2': 0, 'v1': 2, 'v2': -2, 'c': 0}

# D7.E1 — le réseau du collègue (exercices 7.1 et 7.2)
collegue = {'w11': 1, 'w12': 1, 'b1': -4, 'w21': -1, 'w22': 2, 'b2': -1, 'v1': 2, 'v2': 1, 'c': -2}

# D5.1 — les douze messages du chapitre 5 : étiquette, liens, mots suspects
douze = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'etiquette', 'liens', 'mots_suspects'])

M7 = (2, 3)   # le message M7 : 2 liens, 3 mots suspects, spam (y = 1)
M3 = (4, 0)   # la lettre d'information M3 : 4 liens, 0 mot suspect, légitime (y = 0)

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 7.1 à 7.3.

In [ ]:
# Leçon 7.1 — trajet aller du message M7 (équations 7.1 et 7.2)
v = aller(reseau, M7)
print(f"N1 : z1 = {v['z1']:.3f}, a1 = {v['a1']:.4f}")
print(f"N2 : z2 = {v['z2']:.3f}, a2 = {v['a2']:.4f}")
print(f"sortie : z = {v['z']:.3f}, p̂ = {v['p']:.3f}, perte = {perte(1, v['p']):.3f}")
print(f"M3 (lettre d'information) : a2 = {aller(reseau, M3)['a2']:.3f}, p̂ = {aller(reseau, M3)['p']:.3f}")
dessiner(reseau, M7)

In [ ]:
# Leçon 7.2 — la règle de la chaîne pour w11 (équation 7.3), facteur par facteur
f1 = v['p'] - 1                  # ① perte ← z : p̂ − y
f2 = reseau['v1']                # ② z ← a1 : v1
f3 = v['a1'] * (1 - v['a1'])     # ③ a1 ← z1 : dérivée de la sigmoïde, avec l'ACTIVATION a1
f4 = M7[0]                       # ④ z1 ← w11 : x1
d_w11 = f1 * f2 * f3 * f4
print(f"① {f1:.3f}  ② {f2:g}  ③ {f3:.4f}  ④ {f4}  → dérivée = {d_w11:.3f}")

# Vérification par une petite perturbation numérique (comme les sécantes de la leçon 3.3)
def perte_w11(w):
    return perte(1, aller(dict(reseau, w11=w), M7)['p'])
h = 0.0001
print(f"dérivée numérique : ({perte_w11(0.5 + h):.6f} − {perte_w11(0.5 - h):.6f}) / (2h) = "
      f"{(perte_w11(0.5 + h) - perte_w11(0.5 - h)) / (2 * h):.4f}")

# Une mise à jour à la main, η = 0,5, puis on refait le trajet aller
eta = 0.5
w11_nouveau = round(0.5 - eta * d_w11, 3)   # arrondi au millième, comme dans le livre : 0,655
v_n = aller(dict(reseau, w11=w11_nouveau), M7)
print(f"w11 : 0,5 → {w11_nouveau:.3f} ; z1 = {v_n['z1']:.2f}, a1 = {v_n['a1']:.4f}, z = {v_n['z']:.3f}, "
      f"p̂ = {v_n['p']:.3f}, perte = {perte(1, v_n['p']):.3f}")
dessiner(reseau, M7, retour=True)

ws = np.linspace(-1, 3, 200)
plt.figure(figsize=(6, 3))
plt.plot(ws, [perte_w11(w) for w in ws], color='dimgrey', label='perte de M7 selon w11')
plt.plot(ws[25:126], perte_w11(0.5) + d_w11 * (ws[25:126] - 0.5), '--', color=RETOUR, label='tangente en 0,5')
plt.scatter([0.5, w11_nouveau], [perte_w11(0.5), perte_w11(w11_nouveau)], color=['black', AVANT], zorder=3)
plt.xlabel('w11'); plt.ylabel('perte de M7'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Leçon 7.3 — rétropropagation : les neuf dérivées (équation 7.4), puis une mise à jour de TOUS les paramètres
def retour(P, x, y):
    v = aller(P, x)
    d = v['p'] - y                                   # signal d'erreur de la sortie
    d1 = d * P['v1'] * v['a1'] * (1 - v['a1'])       # signal d'erreur de N1
    d2 = d * P['v2'] * v['a2'] * (1 - v['a2'])       # signal d'erreur de N2
    return {'w11': d1 * x[0], 'w12': d1 * x[1], 'b1': d1, 'w21': d2 * x[0], 'w22': d2 * x[1], 'b2': d2,
            'v1': d * v['a1'], 'v2': d * v['a2'], 'c': d}


def mise_a_jour(P, g, eta):
    return {k: P[k] - eta * g[k] for k in P}        # toutes les dérivées sont prises AVANT de modifier


g = retour(reseau, M7, 1)
nouveau = mise_a_jour(reseau, g, 0.5)
print(pd.DataFrame({'valeur': reseau, 'dérivée': g, 'nouvelle valeur': nouveau}).round(3))
print(f"M7 : p̂ = {aller(nouveau, M7)['p']:.3f}, perte = {perte(1, aller(nouveau, M7)['p']):.3f}")
print(f"M3 avant : p̂ = {aller(reseau, M3)['p']:.3f}, perte = {perte(0, aller(reseau, M3)['p']):.3f} ; "
      f"après : p̂ = {aller(nouveau, M3)['p']:.3f}, perte = {perte(0, aller(nouveau, M3)['p']):.3f}")

## Atelier facultatif — entraîner le réseau sur les douze messages

1 000 pas de descente de gradient ($\eta = 0{,}5$), à partir des poids de la leçon 7.1, en moyennant à chaque pas les dérivées des douze messages du chapitre 5. **Toutes les mesures ci-dessous portent sur ces douze messages, qui servent à l'entraînement** : elles ne disent rien du comportement du réseau sur des messages nouveaux (leçon 6.3).

In [ ]:
# Atelier facultatif : entraînement sur les douze messages (mesures sur les données d'entraînement)
X12 = list(zip(douze['liens'], douze['mots_suspects']))
y12 = [1 if e == 'spam' else 0 for e in douze['etiquette']]


def perte_12(P):
    return np.mean([perte(y, aller(P, x)['p']) for x, y in zip(X12, y12)])


def gradient_12(P):
    gs = [retour(P, x, y) for x, y in zip(X12, y12)]
    return {k: np.mean([g[k] for g in gs]) for k in P}     # dérivée de la perte moyenne = moyenne des dérivées


P_entraine, suivi = dict(reseau), [perte_12(reseau)]
for _ in range(1000):
    P_entraine = mise_a_jour(P_entraine, gradient_12(P_entraine), 0.5)
    suivi.append(perte_12(P_entraine))
print(f"perte moyenne sur les douze messages d'entraînement : avant {suivi[0]:.3f}, après {suivi[-1]:.3f}")
for nom, x in (('M7', M7), ('M3', M3)):
    print(f"{nom} : p̂ avant {aller(reseau, x)['p']:.3f}, après {aller(P_entraine, x)['p']:.3f}")
print('erreurs au seuil 0,5 sur les douze : avant', sum((aller(reseau, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)),
      ', après', sum((aller(P_entraine, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)))
print('poids après entraînement :', {k: round(float(v), 2) for k, v in P_entraine.items()})
plt.figure(figsize=(6, 3)); plt.plot(suivi, color=AVANT)
plt.xlabel('pas'); plt.ylabel("perte moyenne (douze messages d'entraînement)"); plt.grid(alpha=0.4); plt.show()

## Exercice 7.1 — Lire le réseau d'un collègue

*Notions : couches, poids et biais (leçon 7.1, équations 7.1 et 7.2).* Données D7.E1 : le dictionnaire `collegue`.

### a) à e) Corrigé commenté

**a)** Six poids dans la couche cachée ($w_{11}$, $w_{12}$, $w_{21}$, $w_{22}$) et en sortie ($v_1$, $v_2$), trois biais ($b_1$, $b_2$, $c$) : neuf paramètres, comme le réseau de la leçon 7.1.

**b)** $w_{21} = -1$ relie les liens au neurone N2 ; $w_{12} = 1$ relie les mots suspects au neurone N1.

**c)** Le biais −4 est celui de N1. Le biais de la sortie vaut $c = -2$.

**d)** Avec $x_1 = x_2 = 0$, les produits par les poids sont nuls : $z_1 = b_1 = -4$ et $z_2 = b_2 = -1$. Le biais est la somme pondérée d'un message « vide ».

**e)** Ici, N2 calcule $-x_1 + 2x_2 - 1$ : il grandit avec les mots suspects et diminue avec les liens, et son poids de sortie $v_2 = 1$ est positif : quand son activation augmente, la probabilité de spam **monte**. Dans le réseau de la leçon 7.1, N2 calculait $x_1 - x_2$ et son poids de sortie était négatif : c'est presque l'effet inverse.

In [ ]:
nb_parametres = len(collegue)
vide = aller(collegue, (0, 0))

In [ ]:
# Vérification de a) et d)
if nb_parametres is None or vide is None:
    print('pas encore complété. Complétez les deux lignes.')
else:
    print(f"{nb_parametres} paramètres ; message vide : z1 = {vide['z1']:g}, z2 = {vide['z2']:g} →",
          'correct' if nb_parametres == 9 and vide['z1'] == -4 and vide['z2'] == -1 else 'à revoir')

## Exercice 7.2 — Calculer une sortie à la main

*Notions : trajet aller (équations 7.1 et 7.2), sigmoïde (équation 5.4), perte (équation 5.6), règle de décision (équation 5.1).*

### a) à c) Corrigé commenté

**a)** M6 : $z_1 = 1 \times 2 + 1 \times 2 - 4 = 0$, $a_1 = \sigma(0) = 0{,}5$ ; $z_2 = -1 \times 2 + 2 \times 2 - 1 = 1$, $a_2 = \sigma(1) \approx 0{,}7311$ ; $z = 2 \times 0{,}5 + 1 \times 0{,}7311 - 2 \approx -0{,}269$ ; $\hat p = \sigma(-0{,}269) \approx 0{,}433$.

**b)** $0{,}433 < 0{,}5$ : M6 n'est pas signalé, ce qui est juste puisqu'il est légitime. Sa perte vaut $-\ln(1 - 0{,}433) \approx 0{,}568$.

**c)** M7 : $z_1 = 2 + 3 - 4 = 1$, $a_1 \approx 0{,}7311$ ; $z_2 = -2 + 6 - 1 = 3$, $a_2 \approx 0{,}9526$ ; $z = 1{,}4621 + 0{,}9526 - 2 \approx 0{,}415$ ; $\hat p \approx 0{,}602$, perte $-\ln(0{,}602) \approx 0{,}507$. Le réseau de la leçon 7.1 fait payer à M7 une perte plus petite, 0,401 : il lui donne une probabilité plus proche de 1.

In [ ]:
p_M6 = aller(collegue, (2, 2))['p']
perte_M6 = perte(0, p_M6)                          # M6 est légitime : y = 0
perte_M7 = perte(1, aller(collegue, M7)['p'])      # M7 est un spam : y = 1

In [ ]:
# Vérification de a) à c)
if p_M6 is None or perte_M6 is None or perte_M7 is None:
    print('pas encore complété. Complétez les trois lignes.')
else:
    print(f"M6 : p̂ = {p_M6:.3f}, signalé : {p_M6 >= 0.5}, perte = {perte_M6:.3f} →",
          'correct' if abs(p_M6 - 0.4332) < 5e-4 and abs(perte_M6 - 0.5677) < 5e-4 else 'à revoir')
    print(f"M7 : perte {perte_M7:.3f} (collègue) contre {perte(1, aller(reseau, M7)['p']):.3f} (leçon 7.1) →",
          'correct' if abs(perte_M7 - 0.5071) < 5e-4 else 'à revoir')
    dessiner(collegue, (2, 2))

## Exercice 7.3 — Interpréter une activation

*Notions : rôle d'un neurone caché (leçon 7.1).* Réseau de la leçon 7.1, douze messages du chapitre 5.

### a) à d) Corrigé commenté

**a)** $a_2$ dépasse 0,8 pour M2 (0,881) et M3 (0,982) : deux messages légitimes, avec des liens et **aucun** mot suspect. $a_2$ vaut exactement 0,5 quand $z_2 = x_1 - x_2 = 0$, c'est-à-dire quand le message a autant de liens que de mots suspects : M4, M6, M9.

**b)** N2 s'allume (activation au-dessus de 0,5) quand un message a plus de liens que de mots suspects, et d'autant plus que l'écart est grand. Son poids de sortie $v_2 = -2$ fait alors baisser le score de sortie, donc la probabilité de spam. Sur ces six messages, les deux où N2 s'allume le plus sont légitimes ; mais six messages, avec des poids choisis à la main, ne suffisent pas pour dire que N2 « repère les lettres d'information » : il faudrait le vérifier sur beaucoup de messages nouveaux.

**c)** M6 : $z = 2 \times 0{,}5 - 2 \times 0{,}5 + 0 = 0$ exactement, donc $\hat p = 0{,}5$ exactement. Avec la règle « supérieur ou égal », M6 est signalé : une fausse alerte, au bord du seuil. C'est la situation de M3 au chapitre 5, dont le score brut valait exactement 0.

**d)** C'est N2 : avec $a_2 \approx 0{,}982$, il retire environ 1,96 au score de sortie, et le réseau ne donne plus que 0,276 à M3. Une activation n'est pas une probabilité de spam : c'est un nombre entre 0 et 1 qui sert d'ingrédient à la sortie. Seul $\hat p$, en sortie, est une probabilité estimée.

In [ ]:
douze['a2'] = [aller(reseau, (l, m))['a2'] for l, m in zip(douze['liens'], douze['mots_suspects'])]
allumes = list(douze.loc[douze['a2'] > 0.8, 'message'])

In [ ]:
# Vérification de a) et c)
if allumes is None or douze['a2'].isna().any():
    print('pas encore complété. Complétez les deux lignes.')
else:
    douze['a1'] = [aller(reseau, (l, m))['a1'] for l, m in zip(douze['liens'], douze['mots_suspects'])]
    douze['p̂'] = [aller(reseau, (l, m))['p'] for l, m in zip(douze['liens'], douze['mots_suspects'])]
    print(douze.round(3).to_string(index=False))
    print('a2 > 0,8 :', allumes, '→', 'correct' if allumes == ['M2', 'M3'] else 'à revoir')
    print(f"M6 : z = {aller(reseau, (2, 2))['z']}, p̂ = {aller(reseau, (2, 2))['p']} → signalé au seuil 0,5 : {aller(reseau, (2, 2))['p'] >= 0.5}")

## Exercice 7.4 — Compléter une dérivée dans Jupyter

*Notions : règle de la chaîne (équation 7.3), vérification numérique (leçon 7.2).* Le poids $w_{22}$ relie les mots suspects au neurone N2.

### a) et b) Corrigé commenté

**a)** ① $\hat p - y \approx -0{,}3303$ ; ② $v_2 = -2$ ; ③ $a_2(1 - a_2) = 0{,}2689 \times 0{,}7311 \approx 0{,}1966$ ; ④ $x_2 = 3$. Produit : $-0{,}3303 \times (-2) \times 0{,}1966 \times 3 \approx 0{,}390$.

**b)** Le signe est positif : augmenter $w_{22}$ fait **monter** la perte. Suivez les signes : une hausse de $w_{22}$ fait monter $z_2$ (car $x_2 = 3 > 0$), donc $a_2$ ; comme $v_2 < 0$, $z$ baisse ; $\hat p$ baisse, alors que M7 est un spam dont la probabilité est déjà trop basse. Deux facteurs négatifs, ① et ②, donnent un produit positif.

In [ ]:
def derivee_w22(x, y):
    v = aller(reseau, x)
    return (v['p'] - y) * reseau['v2'] * v['a2'] * (1 - v['a2']) * x[1]   # ① × ② × ③ × ④

In [ ]:
# Vérification de c) : règle de la chaîne contre perturbation numérique
if derivee_w22(M7, 1) is None:
    print('pas encore complété. Complétez la fonction.')
else:
    h = 0.0001
    num = (perte(1, aller(dict(reseau, w22=-1 + h), M7)['p']) - perte(1, aller(dict(reseau, w22=-1 - h), M7)['p'])) / (2 * h)
    print(f"règle de la chaîne : {derivee_w22(M7, 1):.4f} →", 'correct' if abs(derivee_w22(M7, 1) - 0.3896) < 5e-4 else 'à revoir')
    print(f"dérivée numérique  : {num:.4f} → écart {abs(num - derivee_w22(M7, 1)):.1e} →",
          'correct' if abs(num - derivee_w22(M7, 1)) < 1e-6 else 'à revoir')

## Exercice 7.5 — Prédire, puis vérifier : l'effet d'un changement de poids

*Notions : dérivée et approximation (leçons 3.3 et 7.2).*

### a) Corrigé commenté

Avec la dérivée seule : perte $\approx 0{,}401 - 0{,}310 \times \Delta w$. Pour $\Delta w = 0{,}1$ : $0{,}401 - 0{,}031 = 0{,}370$. Pour $\Delta w = 2$ : $0{,}401 - 0{,}620 \approx -0{,}22$, une perte **négative**, ce qui est impossible.

In [ ]:
def perte_apres(dw):
    return perte(1, aller(dict(reseau, w11=0.5 + dw), M7)['p'])   # on refait tout le trajet aller

In [ ]:
# Vérification de b)
if perte_apres(0.1) is None:
    print('pas encore complété. Complétez la fonction.')
else:
    L0 = perte(1, aller(reseau, M7)['p'])
    for dw in (0.1, 2):
        print(f"dw = {dw} : prédite {L0 + d_w11 * dw:.3f}, exacte {perte_apres(dw):.3f}")
    print('→', 'correct' if abs(perte_apres(0.1) - 0.3716) < 5e-4 and abs(perte_apres(2) - 0.2126) < 5e-4 else 'à revoir')

### b) et c) Corrigé commenté

Pertes exactes : 0,372 pour $w_{11} = 0{,}6$, très proche des 0,370 prédits ; 0,213 pour $w_{11} = 2{,}5$, très loin des −0,22 prédits. La dérivée décrit la courbe **près** du point de départ : la tangente colle à la courbe pour un petit pas, puis s'en écarte (figure de la leçon 7.2). Pour un grand pas, la perte s'aplatit et ne peut pas devenir négative. C'est pourquoi le taux d'apprentissage reste petit : chaque pas doit rester dans la zone où la dérivée dit vrai, quitte à en faire beaucoup.

## Exercice 7.6 — Défi : mettre à jour plusieurs poids sur deux messages

*Notions : rétropropagation (équation 7.4), perte moyenne (équation 5.7), descente de gradient (équation 3.7).* Messages M7 (spam) et M3 (légitime).

### a) à c) Corrigé commenté

**a)** Pertes de départ : 0,401 pour M7 et $-\ln(1 - 0{,}2761) \approx 0{,}323$ pour M3 ; moyenne $\approx 0{,}362$.

**b)** $\delta_{M7} = 0{,}6697 - 1 \approx -0{,}330$ et $\delta_{M3} = 0{,}2761 - 0 = 0{,}2761$. Dérivées de la perte moyenne : $\partial L/\partial v_1 = (-0{,}330 \times 0{,}6225 + 0{,}2761 \times 0{,}5)/2 \approx -0{,}034$ ; $\partial L/\partial v_2 = (-0{,}330 \times 0{,}2689 + 0{,}2761 \times 0{,}9820)/2 \approx 0{,}091$ ; $\partial L/\partial c = (-0{,}330 + 0{,}2761)/2 \approx -0{,}027$.

**c)** Avec $\eta = 1$ : $v_1 \approx 2{,}034$, $v_2 \approx -2{,}091$, $c \approx 0{,}027$. Les activations cachées ne changent pas. Nouvelles pertes : environ 0,393 pour M7 et 0,311 pour M3, moyenne 0,352. La perte moyenne baisse, et **chacune** des deux aussi : contrairement à la mise à jour sur M7 seul (leçon 7.3), M3 n'est pas sacrifié, parce que sa dérivée entre dans la moyenne.

In [ ]:
def gradient_moyen(P):
    g7, g3 = retour(P, M7, 1), retour(P, M3, 0)
    return {k: (g7[k] + g3[k]) / 2 for k in P}    # dérivée d'une moyenne = moyenne des dérivées

In [ ]:
# Vérification de c), puis d) : les neuf paramètres, un pas puis 100 pas
def perte_moyenne(P):
    return (perte(1, aller(P, M7)['p']) + perte(0, aller(P, M3)['p'])) / 2

if gradient_moyen(reseau) is None:
    print('pas encore complété. Complétez la fonction.')
else:
    g = gradient_moyen(reseau)
    sortie_seule = dict(reseau, v1=reseau['v1'] - g['v1'], v2=reseau['v2'] - g['v2'], c=reseau['c'] - g['c'])
    print(f"départ : perte moyenne {perte_moyenne(reseau):.3f}")
    print(f"c) v1, v2, c seulement : M7 {perte(1, aller(sortie_seule, M7)['p']):.3f}, "
          f"M3 {perte(0, aller(sortie_seule, M3)['p']):.3f}, moyenne {perte_moyenne(sortie_seule):.3f} →",
          'correct' if abs(perte_moyenne(sortie_seule) - 0.3519) < 5e-4 else 'à revoir')
    neuf = mise_a_jour(reseau, g, 1)
    print(f"d) les neuf paramètres, un pas : moyenne {perte_moyenne(neuf):.3f} →",
          'correct' if abs(perte_moyenne(neuf) - 0.2584) < 5e-4 else 'à revoir')
    P, suivi = dict(reseau), []
    for _ in range(100):
        P = mise_a_jour(P, gradient_moyen(P), 1)
        suivi.append(perte_moyenne(P))
    print(f"   100 pas : p̂(M7) = {aller(P, M7)['p']:.3f}, p̂(M3) = {aller(P, M3)['p']:.3f}, perte moyenne {suivi[-1]:.3f}")
    plt.figure(figsize=(6, 3)); plt.plot(range(1, 101), suivi, color=AVANT)
    plt.xlabel('pas'); plt.ylabel('perte moyenne (M7 et M3)'); plt.grid(alpha=0.4); plt.show()

### d) Corrigé commenté

Un pas sur les neuf paramètres fait tomber la perte moyenne de 0,362 à 0,258 : bien plus que la couche de sortie seule (0,352), parce que les neurones cachés s'adaptent aussi. Après 100 pas, le réseau donne 0,982 à M7 et 0,018 à M3 : perte moyenne 0,018. Ce succès ne prouve rien pour la suite : avec neuf paramètres pour deux messages, le réseau peut les apprendre par cœur, comme le filtre B de la leçon 6.3. Seuls des messages qu'il n'a pas vus diraient ce qu'il vaut.